# 01 — Data Cleaning

Source: `Maji_Ndogo_farm_survey_small.db` (4 tables, 5,654 agricultural fields: geographic,
weather, soil/crop, and farm-management features) plus a separate `Weather_Station_Dataset.zip`
containing raw, unstructured weather-station text messages that need to be parsed before use.

**This notebook's main finding: `Crop_type` and `Annual_yield` are swapped in the source
schema.** The column labeled `Crop_type` contains continuous numbers; the column labeled
`Annual_yield` contains crop-name strings. This is caught, verified, and fixed below — building
a yield-prediction model on the mislabeled columns would have meant training on the wrong target
entirely.


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import re

pd.set_option("display.width", 140)

DB = "C:/Users/Administrator/Downloads/ALX Coursework/Datasets/Maji_Ndogo_farm_survey_small.db"
conn = sqlite3.connect(DB)
geo = pd.read_sql("SELECT * FROM geographic_features", conn)
weather = pd.read_sql("SELECT * FROM weather_features", conn)
soil = pd.read_sql("SELECT * FROM soil_and_crop_features", conn)
farm = pd.read_sql("SELECT * FROM farm_management_features", conn)
conn.close()

for name, df in [("geographic", geo), ("weather", weather), ("soil_and_crop", soil), ("farm_management", farm)]:
    print(f"{name}: {df.shape}, duplicate Field_IDs: {df['Field_ID'].duplicated().sum()}, nulls: {df.isna().sum().sum()}")


geographic: (5654, 6), duplicate Field_IDs: 0, nulls: 0
weather: (5654, 5), duplicate Field_IDs: 0, nulls: 0
soil_and_crop: (5654, 4), duplicate Field_IDs: 0, nulls: 0
farm_management: (5654, 6), duplicate Field_IDs: 0, nulls: 0


## The Crop_type / Annual_yield swap

In [2]:
print("dtype of 'Crop_type':", farm["Crop_type"].dtype)
print("dtype of 'Annual_yield':", farm["Annual_yield"].dtype)
print()
print("Sample 'Annual_yield' values (should be numeric, are actually crop names):")
print(farm["Annual_yield"].head(5).tolist())
print()
print("'Crop_type' summary stats (should be categorical, is actually a continuous yield number):")
print(farm["Crop_type"].describe())


dtype of 'Crop_type': float64
dtype of 'Annual_yield': str

Sample 'Annual_yield' values (should be numeric, are actually crop names):
['cassava', 'cassava', 'tea', 'cassava', 'wheat']

'Crop_type' summary stats (should be categorical, is actually a continuous yield number):
count    5654.000000
mean        2.090399
std         1.935000
min         0.172226
25%         0.767032
50%         1.446648
75%         2.530662
max        12.695224
Name: Crop_type, dtype: float64


Confirmed: `Annual_yield` holds strings like `"cassava"`, `"wheat "`, `"wheatn"` — crop names,
some with whitespace or typo noise. `Crop_type` holds a continuous value (mean ≈ 2.09, range
0.17–12.7) — this is the real annual yield. The two columns are swapped. Fixed below, along with
cleaning the crop-name typos found in the process.

In [3]:
farm = farm.rename(columns={"Crop_type": "Annual_yield_fixed", "Annual_yield": "Crop_type_fixed"})
farm = farm.rename(columns={"Annual_yield_fixed": "Annual_yield", "Crop_type_fixed": "Crop_type"})

print("Raw crop name values (post-swap-fix, pre-typo-cleaning):")
print(farm["Crop_type"].value_counts())


Raw crop name values (post-swap-fix, pre-typo-cleaning):
Crop_type
wheat       1316
tea          895
potato       823
cassava      660
banana       633
coffee       607
maize        399
rice         265
wheat         13
wheatn        13
tea            9
teaa           9
cassaval       6
cassava        6
Name: count, dtype: int64


In [4]:
farm["Crop_type"] = farm["Crop_type"].str.strip().str.lower()
typo_map = {"wheatn": "wheat", "teaa": "tea", "cassaval": "cassava"}
farm["Crop_type"] = farm["Crop_type"].replace(typo_map)

print("Cleaned crop_type values:")
print(farm["Crop_type"].value_counts())
print()
print("Annual_yield (numeric) summary after fix:")
print(farm["Annual_yield"].describe())


Cleaned crop_type values:
Crop_type
wheat      1342
tea         913
potato      823
cassava     672
banana      633
coffee      607
maize       399
rice        265
Name: count, dtype: int64

Annual_yield (numeric) summary after fix:
count    5654.000000
mean        2.090399
std         1.935000
min         0.172226
25%         0.767032
50%         1.446648
75%         2.530662
max        12.695224
Name: Annual_yield, dtype: float64


## Joining the four tables

In [5]:
df = geo.merge(weather, on="Field_ID").merge(soil, on="Field_ID").merge(farm, on="Field_ID")
print(df.shape)
df.head(3)


(5654, 18)


In [6]:
print("Range checks:")
print("Soil_fertility:", df["Soil_fertility"].min(), "-", df["Soil_fertility"].max())
print("pH:", df["pH"].min(), "-", df["pH"].max())
print("Pollution_level:", df["Pollution_level"].min(), "-", df["Pollution_level"].max())
print("Elevation:", df["Elevation"].min(), "-", df["Elevation"].max())
print("Plot_size:", df["Plot_size"].min(), "-", df["Plot_size"].max())
print()
print("Soil_type values:", df["Soil_type"].unique().tolist())
print("Location (province) values:", df["Location"].unique().tolist())


Range checks:
Soil_fertility: 0.51 - 0.82
pH: 3.700337047698453 - 7.499959152017819
Pollution_level: 1.0132452836581183e-10 - 0.9954133959472448
Elevation: -878.8608 - 1122.2521
Plot_size: 0.5 - 15.0

Soil_type values: ['Sandy', 'Volcanic', 'Loamy', 'Silt', 'Peaty', 'Rocky']
Location (province) values: ['Rural_Akatsi', 'Rural_Sokoto', 'Rural_Kilimani', 'Rural_Hawassa', 'Rural_Amanzi']


## Parsing the weather station text messages

`Weather_station_data.csv` (1,843 rows) holds free-text sensor messages. A first pass with 4
regex patterns only matched 744 of 1,843 messages (40%) — rather than accept that silently, the
unmatched messages were inspected directly, which revealed **10 distinct templates, not 4**:
three metrics (temperature, rainfall, pollution/air-quality), each with 3-4 English and Chinese
phrasings, e.g.:

```
【2022-01-04 21:47:48】温度感应: 现在温度是 12.82C.
Temperature Read at [2022-01-08 02:54:10]: 12.75C.
2023-07-09 22:14:03, Detected 1245.45 mm rainfall in the last 24 hours.
2022-05-13 01:35:33 雨量警告: 764.08 mm.
Air Quality (2022-09-26 16:17:27): Pollution at 0.06.
Pollution Index on 2022-07-03 02:50:56 = -0.15.
```

All 10 templates are matched below, aggregated into structured (station, metric, value)
records, then compared against the clean field-level `weather_features` table as a sanity check.

In [7]:
import zipfile, io

with zipfile.ZipFile("C:/Users/Administrator/Downloads/ALX Coursework/Datasets/Weather_Station_Dataset.zip") as z:
    station_msgs = pd.read_csv(io.BytesIO(z.read("Weather_station_data.csv")))
    field_mapping = pd.read_csv(io.BytesIO(z.read("Weather_data_field_mapping.csv"))).drop(columns=["Unnamed: 0"], errors="ignore")

print(station_msgs.shape, field_mapping.shape)
station_msgs.head(3)


(1843, 2) (5654, 2)


In [8]:
NUM = r"(-?\d+\.?\d*)"

TEMPERATURE_PATTERNS = [
    re.compile(r"现在温度是\s*" + NUM + r"C"),
    re.compile(r"Temperature Read at .*?:\s*" + NUM + r"C"),
    re.compile(r"Current\s*" + NUM + r"\s*C"),
]
RAINFALL_PATTERNS = [
    re.compile(r"Detected\s*" + NUM + r"\s*mm rainfall"),
    re.compile(r"rainfall stands at\s*" + NUM + r"\s*mm"),
    re.compile(r"雨量警告:\s*" + NUM + r"\s*mm"),
]
RAIN_SENSOR_RE = re.compile(NUM + r"\s*mm precipitation detected")
POLLUTION_PATTERNS = [
    re.compile(r"Pollution at\s*" + NUM),
    re.compile(r"Air Quality Index\s*=\s*" + NUM),
    re.compile(r"Pollution Index on.*?=\s*" + NUM),
]

def parse_message(msg):
    for pat in TEMPERATURE_PATTERNS:
        m = pat.search(msg)
        if m:
            return "temperature_c", float(m.group(1))
    for pat in RAINFALL_PATTERNS:
        m = pat.search(msg)
        if m:
            return "rainfall_mm", float(m.group(1))
    m = RAIN_SENSOR_RE.search(msg)
    if m:
        return "rainfall_mm", float(m.group(1))
    for pat in POLLUTION_PATTERNS:
        m = pat.search(msg)
        if m:
            return "pollution_index", float(m.group(1))
    return None, None

parsed = station_msgs["Message"].apply(parse_message)
station_msgs["metric"] = [p[0] for p in parsed]
station_msgs["value"] = [p[1] for p in parsed]

print(f"Parsed {station_msgs['metric'].notna().sum()} / {len(station_msgs)} messages ({station_msgs['metric'].notna().mean():.1%})")
print(station_msgs["metric"].value_counts())


Parsed 1843 / 1843 messages (100.0%)
metric
rainfall_mm        625
temperature_c      611
pollution_index    607
Name: count, dtype: int64


In [9]:
station_rainfall = station_msgs[station_msgs["metric"] == "rainfall_mm"].groupby("Weather_station_ID")["value"].mean()
print("Average parsed rainfall (mm) by station, from free-text messages:")
print(station_rainfall.round(1))

# Cross-check: fields mapped to each station, compared to their OWN field-level
# Rainfall value in the clean weather_features table
field_station = field_mapping.merge(df[["Field_ID", "Rainfall"]], on="Field_ID")
field_avg_by_station = field_station.groupby("Weather_station")["Rainfall"].mean()
print()
print("Average field-level Rainfall (clean table) by mapped station:")
print(field_avg_by_station.round(1))


Average parsed rainfall (mm) by station, from free-text messages:
Weather_station_ID
0    1576.0
1     577.4
2    1691.0
3     905.2
4    1200.2
Name: value, dtype: float64

Average field-level Rainfall (clean table) by mapped station:
Weather_station
0    1522.9
1     568.4
2    1705.8
3     931.6
4    1169.0
Name: Rainfall, dtype: float64


**The two independent sources agree closely** (e.g. station 2: 1,691mm parsed from free text vs.
1,706mm from the clean field-level table; station 1: 577mm vs. 568mm) — a solid sanity check that
the regex extraction is correct, even though the two sources measure slightly different things
(noisy point sensor readings vs. field-level annual totals). The text-parsing exercise is kept
here as a data-wrangling demonstration; the clean `weather_features` table (not the parsed
messages) is what's used as the modeling feature below, since it's complete, field-level, and
doesn't depend on regex extraction from unstructured text.

## Export cleaned dataset

In [10]:
df.to_csv("../data/cleaned_farm_data.csv", index=False)
station_msgs[["Weather_station_ID", "metric", "value"]].dropna().to_csv("../data/parsed_weather_messages.csv", index=False)

print(f"Exported cleaned_farm_data.csv: {df.shape}")
print(f"Crops represented: {df['Crop_type'].nunique()} -- {sorted(df['Crop_type'].unique())}")
print(f"Annual_yield range: {df['Annual_yield'].min():.2f} - {df['Annual_yield'].max():.2f}")


Exported cleaned_farm_data.csv: (5654, 18)
Crops represented: 8 -- ['banana', 'cassava', 'coffee', 'maize', 'potato', 'rice', 'tea', 'wheat']
Annual_yield range: 0.17 - 12.70
